In [ ]:
# Example 1: basic get_logger() usage with default configuration.
import io
import structlog

buf = io.StringIO()
structlog.configure(
    processors=[structlog.processors.KeyValueRenderer()],
    logger_factory=structlog.PrintLoggerFactory(file=buf),
)

log = structlog.get_logger()
log.info("user_login", user_id=42, ip="10.0.0.1")
log.warning("rate_limited", endpoint="/api/v1/foo", retries=3)

output = buf.getvalue()
print(output)
assert "user_id=42" in output
assert "endpoint='/api/v1/foo'" in output


In [ ]:
# Example 2: JSONRenderer + processor chain for structured output.
import io
import json
import structlog

buf = io.StringIO()
structlog.configure(
    processors=[
        structlog.processors.add_log_level,
        structlog.processors.TimeStamper(fmt="iso"),
        structlog.processors.JSONRenderer(),
    ],
    logger_factory=structlog.PrintLoggerFactory(file=buf),
)

log = structlog.get_logger()
log.info("order_created", order_id="ord_123", amount_cents=2500)

line = buf.getvalue().strip()
parsed = json.loads(line)
assert parsed["event"] == "order_created"
assert parsed["level"] == "info"
assert parsed["order_id"] == "ord_123"
assert "timestamp" in parsed
print(parsed)


In [ ]:
# Example 3: bound loggers — context binding via .bind() and .new().
import io
import structlog

buf = io.StringIO()
structlog.configure(
    processors=[structlog.processors.KeyValueRenderer(sort_keys=True)],
    logger_factory=structlog.PrintLoggerFactory(file=buf),
)

log = structlog.get_logger()
req_log = log.bind(request_id="req_abc", user="alice")
req_log.info("request_start")
req_log.info("request_end", status=200)

fresh = req_log.new(span="db_query")
fresh.info("query_done", rows=10)

out = buf.getvalue()
print(out)
assert "request_id='req_abc'" in out
assert "span='db_query'" in out
assert "user='alice'" not in out.splitlines()[-1]


In [ ]:
# Example 4: contextvars-based context (thread/async-safe).
import io
import structlog
from structlog.contextvars import (
    bind_contextvars,
    clear_contextvars,
    merge_contextvars,
)

buf = io.StringIO()
structlog.configure(
    processors=[merge_contextvars, structlog.processors.KeyValueRenderer(sort_keys=True)],
    logger_factory=structlog.PrintLoggerFactory(file=buf),
)

clear_contextvars()
bind_contextvars(tenant="acme", trace_id="xyz")

log = structlog.get_logger()
log.info("handler_called", path="/v1/items")
clear_contextvars()
log.info("after_clear")

out = buf.getvalue()
print(out)
assert "tenant='acme'" in out.splitlines()[0]
assert "tenant" not in out.splitlines()[1]


In [ ]:
# Example 5: stdlib integration — wrap_logger + ProcessorFormatter.
import io
import logging
import structlog

buf = io.StringIO()
handler = logging.StreamHandler(buf)
handler.setFormatter(
    structlog.stdlib.ProcessorFormatter(
        processor=structlog.processors.KeyValueRenderer(sort_keys=True),
    )
)
root = logging.getLogger("structlog_test")
root.handlers = [handler]
root.setLevel(logging.INFO)

structlog.configure(
    processors=[
        structlog.stdlib.add_log_level,
        structlog.stdlib.ProcessorFormatter.wrap_for_formatter,
    ],
    logger_factory=structlog.stdlib.LoggerFactory(),
    wrapper_class=structlog.stdlib.BoundLogger,
)

log = structlog.get_logger("structlog_test")
log.info("stdlib_bridge_works", component="db")

out = buf.getvalue()
print(out)
assert "stdlib_bridge_works" in out
assert "component='db'" in out
